In [ ]:
!pip install openai

In [ ]:
import argparse
from openai import OpenAI
import pandas as pd
import time

def get_prediction(claim, reference, model_name):
    prompt_prefix = """
    Your task is to determine whether the reference entail, is unrelated and unverifiable, is related but unverifiable, misinterpret, omit critical information, contain a numeric error, contain an opposite meaning, or contain an entity error to the claim.
    You will be given two inputs: claim, reference.
    Follow this systematic evaluation process:
    Step 1: Paragraph Structure Assessment
    First, determine if the paragraph is well-formed and not a filler paragraph. If the paragraph is poorly structured, contains only filler content, or lacks substantive claims, classify it as N/A.
    Step 2: Verifiability Check
    If the paragraph is well-formed, assess whether it can be verified against the provided reference abstracts. If the content cannot be verified using the reference materials, classify it as Unverifiable.
    Step 3: Relationship Analysis
    For verifiable paragraphs, determine the relationship between the paragraph and the reference abstracts:
    Direct Entailment
    Check if the paragraph is directly supported by at least one passage from the abstracts without contradiction from other passages. If yes, classify as Entailment.
    Direct Contradiction
    If not directly entailed, examine whether the paragraph is directly contradicted by at least one passage from the abstracts. Look for contradictions involving different entities, numeric values, or relations than stated in the abstract. If directly contradicted, classify as Direct Contradiction.
    Indirect Contradiction
    If not directly contradicted, assess whether the paragraph presents logical fallacies or flawed reasoning, including over-claiming, under-claiming, ambiguity, inconsistency, or illogical conclusions. If such issues exist, classify as Indirect Contradiction.
    Step 4: Detailed Contradiction Analysis
    For paragraphs classified as contradictions, identify the specific type:
    Misinterpretation: Logical fallacies or flawed reasoning
    Missing Info: Omits critical parts from abstracts, changing meaning or intent
    Numeric: Contains erroneous numeric values
    Opposite: Negates parts of the abstract or replaces terms with antonyms
    Entity: Contains erroneous entities
    Step 5: Unverifiable Subcategorization
    For unverifiable paragraphs, determine if they relate to the abstracts:
    Related but Unverifiable: Content is related but cannot be verified
    Unrelated and Unverifiable: Content has no connection to the abstracts
    Input:
"""

    client = OpenAI(api_key="")

    full_prompt = f"""
    {prompt_prefix}
    claim: {claim}
    reference: {reference}
    Give the probabilities rounded to 3 decimal places for each category. The total probability is 1. Answer only output format:
    {{
        "Opposite_meaning_probability": Probability to predict class Opposite meaning,
        "Misrepresentation_probability": Probability to predict class Misrepresentation,
        "Related_but_unverifiable_probability": Probability to predict class Related but unverifiable,
        "Entailment_probability": Probability to predict class Entailment,
        "Entity_error_probability": Probability to predict class Entity error,
        "Unrelated_and_unverifiable_probability": Probability to predict class Unrelated and unverifiable,
        "Numeric_error_probability": Probability to predict class Numeric error,
        "Missing_information_probability": Probability to predict class Missing information
    }}
    """
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are an annotator concerned that the claim may not align with the reference."},
                {"role": "user", "content": full_prompt}
            ],
            reasoning_effort="high",
            seed=13
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

def main(output_path, sleep_time, model_name):
    df = pd.read_csv('/content/o3_2.csv')

    predictions = []

    for idx, row in df.iterrows():
        claim = row['claim_clean']
        reference = row['reference_clean']
        prediction = get_prediction(claim, reference, model_name)
        predictions.append(prediction)

        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}...")

        time.sleep(sleep_time)

    df['predict'] = predictions

    df.to_csv(output_path, index=False)

In [ ]:
main("o3-mini-hpct_6.csv", 0.07, 'o3-mini')

Processed 10...
Processed 20...
Processed 30...
Processed 40...
Processed 50...


In [ ]:
import argparse
from openai import OpenAI
import pandas as pd
import time

def get_prediction(claim, reference, model_name):
    prompt_prefix = """
    Your task is to determine whether the reference entail, is unrelated and unverifiable, is related but unverifiable, misinterpret, omit critical information, contain a numeric error, contain an opposite meaning, or contain an entity error to the claim.
    You will be given two inputs: claim, reference.
    Follow this systematic evaluation process:
    Step 1: Paragraph Structure Assessment
    First, determine if the paragraph is well-formed and not a filler paragraph. If the paragraph is poorly structured, contains only filler content, or lacks substantive claims, classify it as N/A.
    Step 2: Verifiability Check
    If the paragraph is well-formed, assess whether it can be verified against the provided reference abstracts. If the content cannot be verified using the reference materials, classify it as Unverifiable.
    Step 3: Relationship Analysis
    For verifiable paragraphs, determine the relationship between the paragraph and the reference abstracts:
    Direct Entailment
    Check if the paragraph is directly supported by at least one passage from the abstracts without contradiction from other passages. If yes, classify as Entailment.
    Direct Contradiction
    If not directly entailed, examine whether the paragraph is directly contradicted by at least one passage from the abstracts. Look for contradictions involving different entities, numeric values, or relations than stated in the abstract. If directly contradicted, classify as Direct Contradiction.
    Indirect Contradiction
    If not directly contradicted, assess whether the paragraph presents logical fallacies or flawed reasoning, including over-claiming, under-claiming, ambiguity, inconsistency, or illogical conclusions. If such issues exist, classify as Indirect Contradiction.
    Step 4: Detailed Contradiction Analysis
    For paragraphs classified as contradictions, identify the specific type:
    Misinterpretation: Logical fallacies or flawed reasoning
    Missing Info: Omits critical parts from abstracts, changing meaning or intent
    Numeric: Contains erroneous numeric values
    Opposite: Negates parts of the abstract or replaces terms with antonyms
    Entity: Contains erroneous entities
    Step 5: Unverifiable Subcategorization
    For unverifiable paragraphs, determine if they relate to the abstracts:
    Related but Unverifiable: Content is related but cannot be verified
    Unrelated and Unverifiable: Content has no connection to the abstracts
    Input:
"""

    client = OpenAI(api_key="")

    full_prompt = f"""
    {prompt_prefix}
    claim: {claim}
    reference: {reference}
    Give the probabilities rounded to 3 decimal places for each category. The total probability is 1. Answer only output format:
    {{
        "Opposite_meaning_probability": Probability to predict class Opposite meaning,
        "Misrepresentation_probability": Probability to predict class Misrepresentation,
        "Related_but_unverifiable_probability": Probability to predict class Related but unverifiable,
        "Entailment_probability": Probability to predict class Entailment,
        "Entity_error_probability": Probability to predict class Entity error,
        "Unrelated_and_unverifiable_probability": Probability to predict class Unrelated and unverifiable,
        "Numeric_error_probability": Probability to predict class Numeric error,
        "Missing_information_probability": Probability to predict class Missing information
    }}
    """
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are an annotator concerned that the claim may not align with the reference."},
                {"role": "user", "content": full_prompt}
            ],
            reasoning_effort="high",
            seed=13
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return None

def main(output_path, sleep_time, model_name):
    df = pd.read_csv('/content/o3_2.csv')

    predictions = []

    for idx, row in df.iterrows():
        claim = row['claim_clean']
        reference = row['reference_clean']
        prediction = get_prediction(claim, reference, model_name)
        predictions.append(prediction)

        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}...")

        time.sleep(sleep_time)

    df['predict'] = predictions

    df.to_csv(output_path, index=False)

In [ ]:
main("o3-mini-hpct_3.csv", 0.07, 'o3-mini')

Processed 10...
Processed 20...
Processed 30...
Processed 40...
Processed 50...


In [ ]:
response = client.chat.completions.create(
            model="o3-mini",
            messages=[
                {"role": "system", "content": "You are an annotator concerned that the claim may not align with the reference."},
                {"role": "user", "content": full_prompt}
            ],
            reasoning_effort="high",
            seed=13
        )

In [ ]:
response.choices[0].message.content

'{\n    "Opposite_meaning_probability": 0.000,\n    "Misrepresentation_probability": 0.800,\n    "Related_but_unverifiable_probability": 0.000,\n    "Entailment_probability": 0.150,\n    "Entity_error_probability": 0.000,\n    "Unrelated_and_unverifiable_probability": 0.000,\n    "Numeric_error_probability": 0.000,\n    "Missing_information_probability": 0.050\n}'